<a href="https://colab.research.google.com/github/buntyedutainer/student-performance-ai/blob/main/Notebook3_Encoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Upload the file into Colab
from google.colab import files
files.upload()   # pick student-mat.csv

Saving student-mat.csv to student-mat.csv


{'student-mat.csv': b'school;sex;age;address;famsize;Pstatus;Medu;Fedu;Mjob;Fjob;reason;guardian;traveltime;studytime;failures;schoolsup;famsup;paid;activities;nursery;higher;internet;romantic;famrel;freetime;goout;Dalc;Walc;health;absences;G1;G2;G3\n"GP";"F";18;"U";"GT3";"A";4;4;"at_home";"teacher";"course";"mother";2;2;0;"yes";"no";"no";"no";"yes";"yes";"no";"no";4;3;4;1;1;3;6;"5";"6";6\n"GP";"F";17;"U";"GT3";"T";1;1;"at_home";"other";"course";"father";1;2;0;"no";"yes";"no";"no";"no";"yes";"yes";"no";5;3;3;1;1;3;4;"5";"5";6\n"GP";"F";15;"U";"LE3";"T";1;1;"at_home";"other";"other";"mother";1;2;3;"yes";"no";"yes";"no";"yes";"yes";"yes";"no";4;3;2;2;3;3;10;"7";"8";10\n"GP";"F";15;"U";"GT3";"T";4;2;"health";"services";"home";"mother";1;3;0;"no";"yes";"yes";"yes";"yes";"yes";"yes";"yes";3;2;2;1;1;5;2;"15";"14";15\n"GP";"F";16;"U";"GT3";"T";3;3;"other";"other";"home";"father";1;2;0;"no";"yes";"yes";"no";"yes";"yes";"no";"no";4;3;2;1;2;5;4;"6";"10";10\n"GP";"M";16;"U";"LE3";"T";4;3;"service

In [2]:
# Cell 1: setup and a look at what needs encoding
import pandas as pd
import numpy as np

df = pd.read_csv("student-mat.csv", sep=";")
cat_cols = df.select_dtypes(exclude="number").columns.tolist()

print("Categorical columns:", len(cat_cols))
for c in cat_cols:
    print(f"{c:12s} {df[c].nunique()} values -> {sorted(df[c].unique())}")

Categorical columns: 17
school       2 values -> ['GP', 'MS']
sex          2 values -> ['F', 'M']
address      2 values -> ['R', 'U']
famsize      2 values -> ['GT3', 'LE3']
Pstatus      2 values -> ['A', 'T']
Mjob         5 values -> ['at_home', 'health', 'other', 'services', 'teacher']
Fjob         5 values -> ['at_home', 'health', 'other', 'services', 'teacher']
reason       4 values -> ['course', 'home', 'other', 'reputation']
guardian     3 values -> ['father', 'mother', 'other']
schoolsup    2 values -> ['no', 'yes']
famsup       2 values -> ['no', 'yes']
paid         2 values -> ['no', 'yes']
activities   2 values -> ['no', 'yes']
nursery      2 values -> ['no', 'yes']
higher       2 values -> ['no', 'yes']
internet     2 values -> ['no', 'yes']
romantic     2 values -> ['no', 'yes']


In [3]:
# Cell 2: split the categorical columns into binary and nominal
binary_cols = [c for c in cat_cols if df[c].nunique() == 2]
nominal_cols = [c for c in cat_cols if df[c].nunique() > 2]

print("Binary :", binary_cols)
print("Nominal:", nominal_cols)

Binary : ['school', 'sex', 'address', 'famsize', 'Pstatus', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']
Nominal: ['Mjob', 'Fjob', 'reason', 'guardian']


In [4]:
# Cell 3: binary encoding with explicit, readable mappings
binary_maps = {
    "school":  {"GP": 0, "MS": 1},
    "sex":     {"F": 0, "M": 1},
    "address": {"R": 0, "U": 1},
    "famsize": {"LE3": 0, "GT3": 1},
    "Pstatus": {"A": 0, "T": 1},
}
yes_no_cols = ["schoolsup", "famsup", "paid", "activities",
               "nursery", "higher", "internet", "romantic"]
for c in yes_no_cols:
    binary_maps[c] = {"no": 0, "yes": 1}

df_enc = df.copy()
for col, mapping in binary_maps.items():
    df_enc[col] = df_enc[col].map(mapping)

# safety check: a wrong mapping would leave NaN behind
print("NaN after binary mapping:", df_enc[list(binary_maps)].isnull().sum().sum())
df_enc[list(binary_maps)].head()

NaN after binary mapping: 0


,school,sex,address,famsize,Pstatus,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic
0,0,0,1,1,0,1,0,0,0,1,1,0,0
1,0,0,1,1,1,0,1,0,0,0,1,1,0
2,0,0,1,0,1,1,0,1,0,1,1,1,0
3,0,0,1,1,1,0,1,1,1,1,1,1,1
4,0,0,1,1,1,0,1,1,0,1,1,0,0


In [5]:
# Cell 4: one-hot encoding for nominal columns
df_enc = pd.get_dummies(df_enc, columns=nominal_cols, drop_first=True, dtype=int)

print("Shape after encoding:", df_enc.shape)
print("New columns:")
print([c for c in df_enc.columns if any(c.startswith(n + "_") for n in nominal_cols)])

Shape after encoding: (395, 42)
New columns:
['Mjob_health', 'Mjob_other', 'Mjob_services', 'Mjob_teacher', 'Fjob_health', 'Fjob_other', 'Fjob_services', 'Fjob_teacher', 'reason_home', 'reason_other', 'reason_reputation', 'guardian_mother', 'guardian_other']


In [6]:
# Cell 5: see what one-hot encoding did to a few rows
before = df[["Mjob", "guardian"]].head(5)
after = df_enc[[c for c in df_enc.columns if c.startswith(("Mjob_", "guardian_"))]].head(5)
print(before, "\n")
print(after)

      Mjob guardian
0  at_home   mother
1  at_home   father
2  at_home   mother
3   health   mother
4    other   father 

   Mjob_health  Mjob_other  Mjob_services  Mjob_teacher  guardian_mother  \
0            0           0              0             0                1   
1            0           0              0             0                0   
2            0           0              0             0                1   
3            1           0              0             0                1   
4            0           1              0             0                0   

   guardian_other  
0               0  
1               0  
2               0  
3               0  
4               0  


In [7]:
# Cell 6: validate, because a model will crash on any leftover text
print("Any non-numeric columns left:", len(df_enc.select_dtypes(exclude="number").columns))
print("Total missing values       :", df_enc.isnull().sum().sum())
print("Final shape (rows, columns):", df_enc.shape)
print("Features (excluding G3)    :", df_enc.shape[1] - 1)

Any non-numeric columns left: 0
Total missing values       : 0
Final shape (rows, columns): (395, 42)
Features (excluding G3)    : 41


In [8]:
# Cell 7: did encoding reveal anything new? Correlation of the new columns with G3
corr_new = df_enc.corr()["G3"].drop("G3").sort_values()

print("Most negative:")
print(corr_new.head(6).round(3))
print("\nMost positive (excluding G1, G2):")
print(corr_new.drop(["G1", "G2"]).tail(6).round(3))

Most negative:
failures     -0.360
age          -0.162
goout        -0.133
romantic     -0.130
traveltime   -0.117
Mjob_other   -0.096
Name: G3, dtype: float64

Most positive (excluding G1, G2):
sex            0.103
address        0.106
Mjob_health    0.116
Fedu           0.152
higher         0.182
Medu           0.217
Name: G3, dtype: float64


In [9]:
# Cell 8: the same job with scikit-learn (this is how it appears in a real pipeline)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

X = df.drop(columns="G3")
y = df["G3"]

preprocess = ColumnTransformer(
    transformers=[
        ("binary", OrdinalEncoder(), binary_cols),
        ("nominal", OneHotEncoder(drop="first", sparse_output=False), nominal_cols),
    ],
    remainder="passthrough",   # keep the numeric columns as they are
)

X_arr = preprocess.fit_transform(X)
print("Shape from the scikit-learn pipeline:", X_arr.shape)
print("Same number of features as the pandas version?", X_arr.shape[1] == df_enc.shape[1] - 1)

Shape from the scikit-learn pipeline: (395, 41)
Same number of features as the pandas version? True


In [10]:
# Cell 9: save the encoded dataset for the next days
df_enc.to_csv("student_encoded.csv", index=False)
print("Saved student_encoded.csv", df_enc.shape)

Saved student_encoded.csv (395, 42)
